# SONARA: Chapter 3 Pipeline (Data Preparation, CNN–BiLSTM, Evaluation, Zone-Level Attribution)

One notebook for the whole offline pipeline. It uses **the same method as `test/SONARA_DATASET_V2.ipynb` and `test/SONARA_TRAIN_V6.ipynb`** (features, augmentation, models, loss, threshold rule, PVSD as external test), but reads the audio straight from this GitHub repo, so nothing has to be set up in Google Drive.

| Notebook part | Manuscript section | Panel rows it supports |
| :- | :- | :- |
| 1. Setup and configuration | — | — |
| 2. Datasets, label consolidation and inventory checks | 3.1.1 | 4 |
| 3. Preprocessing: 16 kHz, mono, the whole 5-s clip per file | 3.1.2, Figure 3.1 | — |
| 4. Features: log-mel (dB) + deltas, 3 × 64 × 500 | 3.1.2, 3.1.3, Figure 3.2 | 4 |
| 5. Splitting (70/15/15, grouped by recording) and augmentation | 3.1.2, 3.1.4 | — |
| 6. CNN–BiLSTM and comparison models | 3.2.1 | 3, 15 |
| 7. Loss, optimizer, scheduler, early stopping | 3.2.2, 3.2.3 | 7 – 10, 18 |
| 8. Training over several seeds | 3.2.3 | 15 |
| 9. Evaluation, baselines and the new Table 3.1 | 3.2.4, 3.3.5 | 11, 12, 15 – 17 |
| 10. External test on PVSD | 3.3.5 | 5 |
| 11. Gateway inference and zone-level attribution (simulation) | 3.3.2 – 3.3.4 | 5 |
| 12. Optional label-smoothing check, and values for the manuscript | 3.2.2 | 7, 8 |

**Where the method differs from the current manuscript text** (the text should be changed to match):

- **The whole 5-second clip is the input, not 1 to 2 second windows.** SmartEars labels describe a whole 5-second clip. A short slice of a "Sick" clip may contain no cough at all, so slicing would create wrong labels.
- **SmartEars only for training, validation and testing.** PVSD is never trained on and is used as an external test, scored per file.
- **Classes are not balanced by removing data.** The real 2 : 1 ratio is kept and handled by the class weights in the loss.
- **Augmentation** is a small random time shift, time masking, frequency masking and a random loudness offset.
- **The decision threshold** is chosen on the validation set to reach a target distress recall. Results at 0.5 are also reported.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all. With 4 models × 5 seeds, expect several hours. Finished runs are saved, so rerunning continues where it stopped.

## 1. Setup and configuration

In [ ]:
import os, re, copy, glob, hashlib, json, random, time
from collections import deque
from math import gcd

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
from scipy.signal import resample_poly

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, roc_curve, confusion_matrix, classification_report,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| torch", torch.__version__)

In [ ]:
# ---- Data location -------------------------------------------------------
REPO_URL = "https://github.com/tagalogclark123-cmyk/SONARA-GITHUB.git"
REPO_DIR = "/content/SONARA-GITHUB"
OUT_DIR = "/content/sonara_outputs"
SAVE_TO_DRIVE = False         # True = keep outputs in Google Drive so a disconnect loses nothing
DRIVE_OUT_DIR = "/content/drive/MyDrive/SONARA/sonara_outputs"

# Each dataset is a folder of class sub-folders inside the repo. The label map sends the
# original labels to three source categories. Datasets whose folder is missing are skipped.
DATASETS = [
    {"name": "SmartEars", "root": "Smartears",
     "label_map": {"Healthy": "Normal", "None": "Noise", "Sick": "Distress"}},
    {"name": "PVSD", "root": "PVSD",
     "label_map": {"Healthy": "Normal", "Noise": "Noise", "Unhealthy": "Distress"}},
]
TRAIN_DATASETS = ["SmartEars"]   # used for train / validation / test
EXTERNAL_DATASETS = ["PVSD"]     # never trained on; external test only

# ---- 3.1.4 Splitting --------------------------------------------------------
SEEDS = [7, 21, 42, 77, 99]   # each seed = one split + one training run per model
SPLIT_FOLDS = (14, 3, 3)      # 20 folds -> 70% train, 15% validation, 15% test
GROUP_BY = "recording"        # "recording" | "device_day" | "device" (kept together in one split)

# ---- 3.1.2 Preprocessing and features ---------------------------------------
SAMPLE_RATE = 16000
CROP_SEC = 5.0                # model input length: the whole SmartEars clip
N_FFT = 400                   # 25 ms STFT window
HOP_LENGTH = 160              # 10 ms hop
N_MELS = 64
MAX_EXTERNAL_WINDOWS = 50     # cap on 5-s windows scored per external file (evenly spaced if more)

# ---- 3.1.2 Augmentation (training split only) -------------------------------
AUG_TIME_MASK_PROB = 0.4      # chance of masking a block of time frames
AUG_FREQ_MASK_PROB = 0.4      # chance of masking a block of mel bands
AUG_GAIN_PROB = 0.2           # chance of a loudness change
AUG_GAIN_DB = (-6.0, 6.0)     # size of the loudness change, in dB
AUG_SHIFT_PROB = 0.5          # chance of a time shift
AUG_SHIFT_SEC = 0.5           # largest time shift, in seconds (0 turns it off)

# ---- 3.2.2 / 3.2.3 Training -------------------------------------------------
MODELS_TO_RUN = ["cnn_bilstm", "cnn_frontend_only", "dscnn", "mobilenetv3_small"]
# Add "cnn_bilstm_laststep" to measure the effect of the BiLSTM pooling fix (old V5 behaviour).
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.02        # epsilon in Equation 3-3
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 8
LR_PATIENCE = 2
LR_FACTOR = 0.5
DROPOUT = 0.3

# ---- 3.3.2 Decision threshold ------------------------------------------------
TARGET_RECALL = 0.90          # threshold = highest one whose validation distress recall is >= this
DEFAULT_THRESHOLD = 0.5       # also reported

# ---- Debugging --------------------------------------------------------------
QUICK_RUN = None              # e.g. 60 = use 60 clips per class, 2 epochs, 2 seeds

CLASS_NAMES = ["Non-distress", "Distress"]
CROP_SAMPLES = int(round(CROP_SEC * SAMPLE_RATE))
N_FRAMES = CROP_SAMPLES // HOP_LENGTH
NUM_WORKERS = 2 if os.name == "posix" else 0
if QUICK_RUN:
    MAX_EPOCHS, SEEDS = 2, SEEDS[:2]
PRIMARY_SEED = 42 if 42 in SEEDS else SEEDS[0]   # the run used for figures, the saved model and the simulation


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = DRIVE_OUT_DIR
RUN_DIR = os.path.join(OUT_DIR, "runs")
os.makedirs(RUN_DIR, exist_ok=True)
print("Outputs:", OUT_DIR)

In [ ]:
# Download the repository (code + SmartEars and PVSD audio, about 3.6 GB). Takes several minutes the first time.
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
print("Repo folder:", REPO_DIR, "| exists:", os.path.isdir(REPO_DIR))

## 2. Datasets, label consolidation and inventory checks (Section 3.1.1)

Original labels are consolidated into two target classes: **Non-distress** (healthy/normal and none/noise) and **Distress** (sick/unhealthy). The three-way source category is kept as a column so that Non-distress errors can be examined separately for normal and noise samples.

The classes are **not** balanced by dropping clips. The counts below are the real ones, and the imbalance is handled by the class weights in the loss (Equation 3-1).

The inventory also records duration, sample rate and an MD5 hash for every file. Read the checks before training: a class whose files are much shorter, or recorded at a different sample rate, could be recognized by that instead of by its sound.

In [ ]:
SMARTEARS_NAME = re.compile(r"^(rpi\d+)-(\d{8})(\d{6})(?:-(\d+))?$")


def md5_of(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def build_manifest():
    rows = []
    for ds in DATASETS:
        root = os.path.join(REPO_DIR, ds["root"])
        if not os.path.isdir(root):
            print(f"[skip] {ds['name']}: folder '{ds['root']}' not found in the repo")
            continue
        for folder, source in ds["label_map"].items():
            for path in sorted(glob.glob(os.path.join(root, folder, "*.wav"))):
                stem = os.path.splitext(os.path.basename(path))[0]
                m = SMARTEARS_NAME.match(stem)
                if m:   # device id + start time identify the source recording
                    device_id, day = m.group(1), m.group(2)
                    recording = f"{device_id}-{day}{m.group(3)}"
                else:
                    device_id, day, recording = stem, stem, stem
                rows.append({
                    "path": path, "dataset": ds["name"], "original_label": folder,
                    "source": source, "label": int(source == "Distress"),
                    "recording": f"{ds['name']}:{recording}",
                    "device_day": f"{ds['name']}:{device_id}-{day}",
                    "device": f"{ds['name']}:{device_id}",
                })
    return pd.DataFrame(rows)


manifest = build_manifest()
assert len(manifest) > 0, "No audio files found. Check REPO_DIR and DATASETS."
if QUICK_RUN:
    manifest = manifest.groupby(["dataset", "source"]).head(QUICK_RUN).reset_index(drop=True)

info = [sf.info(p) for p in manifest["path"]]
manifest["sr"] = [i.samplerate for i in info]
manifest["channels"] = [i.channels for i in info]
manifest["seconds"] = [i.frames / i.samplerate for i in info]
manifest["md5"] = [md5_of(p) for p in manifest["path"]]
# Exact duplicate files must share a split group, otherwise copies could sit in train and test.
for col in ["recording", "device_day", "device"]:
    manifest[col] = manifest.groupby("md5")[col].transform("first")
manifest["role"] = np.where(manifest["dataset"].isin(TRAIN_DATASETS), "train/val/test",
                            np.where(manifest["dataset"].isin(EXTERNAL_DATASETS), "external test", "unused"))
manifest.to_csv(os.path.join(OUT_DIR, "inventory.csv"), index=False)

main_idx = manifest.index[manifest["role"] == "train/val/test"].to_numpy()
external_idx = manifest.index[manifest["role"] == "external test"].to_numpy()

pd.set_option("display.width", 160)
by = ["dataset", "role", "original_label", "source"]
checks = manifest.groupby(by)["seconds"].agg(files="size", min_s="min", median_s="median", max_s="max").round(2)
checks["share_shorter_than_input"] = manifest.assign(short=manifest["seconds"] < CROP_SEC).groupby(by)["short"].mean().round(3)
checks["sample_rates"] = manifest.groupby(by)["sr"].agg(lambda s: sorted(set(s)))
print(checks.to_string())

counts = manifest.loc[main_idx, "label"].value_counts().sort_index()
print(f"\nTarget classes in the training datasets: Non-distress {counts.get(0, 0):,} | Distress {counts.get(1, 0):,}")

dup = manifest[manifest.duplicated("md5", keep=False)]
conflicts = int((dup.groupby("md5")["label"].nunique() > 1).sum()) if len(dup) else 0
print(f"Exact duplicate files: {len(dup)} in {dup['md5'].nunique()} groups | groups with conflicting labels: {conflicts}")

per_recording = manifest.loc[main_idx].groupby("recording").size()
print(f"Source recordings: {len(per_recording):,} | recordings cut into more than one clip: "
      f"{int((per_recording > 1).sum()):,} (up to {int(per_recording.max())} clips each)")

## 3. Preprocessing (Section 3.1.2)

Every recording is resampled to 16 kHz and converted to mono. Each SmartEars clip is then used whole, as **one 5-second input**, because that is the unit the labels describe. Files longer than 5 seconds exist only in the external PVSD set and are cut into 5-second windows there.

In [ ]:
def load_audio(path, target_sr=SAMPLE_RATE):
    """Read a file, average channels to mono, and resample to target_sr."""
    wav, sr = sf.read(path, dtype="float32", always_2d=True)
    wav = wav.mean(axis=1)
    if sr != target_sr:
        g = gcd(int(sr), int(target_sr))
        wav = resample_poly(wav, target_sr // g, sr // g).astype(np.float32)
    return wav, sr


t0 = time.time()
clips = []            # one int16 waveform per file, at 16 kHz mono
for i, path in enumerate(manifest["path"]):
    wav, _ = load_audio(path)
    clips.append(np.clip(wav * 32767.0, -32768, 32767).astype(np.int16))
    if (i + 1) % 500 == 0:
        print(f"  loaded {i + 1}/{len(manifest)} clips ({time.time() - t0:.0f}s)")
print(f"Loaded {len(clips)} clips in {time.time() - t0:.0f}s")


def crop(clip, start=None):
    """5-s input as float32 in [-1, 1], plus its real length in samples. A clip shorter than
    5 s is returned whole with zeros after it; the padded part is fixed later, on the features."""
    if start is None:                                   # centre crop
        start = max(0, (len(clip) - CROP_SAMPLES) // 2)
    seg = clip[start:start + CROP_SAMPLES].astype(np.float32) / 32768.0
    n = len(seg)
    if n < CROP_SAMPLES:
        seg = np.pad(seg, (0, CROP_SAMPLES - n))
    return seg, n

In [ ]:
# Figure 3.1: example waveform before and after resampling to 16 kHz mono.
example_path = manifest.loc[manifest["label"] == 1, "path"].iloc[0]
raw, raw_sr = sf.read(example_path, dtype="float32", always_2d=True)
resampled, _ = load_audio(example_path)

fig, axes = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
axes[0].plot(np.arange(len(raw)) / raw_sr, raw[:, 0], linewidth=0.5)
axes[0].set_title(f"Original: {raw_sr} Hz, {raw.shape[1]} channel(s)")
axes[1].plot(np.arange(len(resampled)) / SAMPLE_RATE, resampled, linewidth=0.5)
axes[1].set_title(f"After preprocessing: {SAMPLE_RATE} Hz, mono")
axes[1].set_xlabel("Time (s)")
for ax in axes:
    ax.set_ylabel("Amplitude")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figure_3_1_resampling.png"), dpi=200)
plt.show()

## 4. Feature representation (Sections 3.1.2 and 3.1.3)

Each 5-second clip is passed through a short-time Fourier transform (25 ms window, 10 ms hop) and a 64-band mel filterbank, and the power in each band is converted to decibels. The log-mel matrix and its first and second derivatives (delta and delta-delta) are stacked into a **3 × 64 × 500 tensor**.

A clip shorter than 5 seconds is padded on the features: the dB channel with that clip's own minimum value and the delta channels with 0. Padding with 0 dB would insert a loud block that a model could learn as a shortcut. Some SmartEars clips are a few milliseconds short of 5 seconds (the shortest is 4.985 s), so they get one or two padded frames; the short PVSD files get more.

In [ ]:
class LogMelDeltas(nn.Module):
    """Waveform (batch, samples) -> feature tensor (batch, 3, n_mels, frames)."""

    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
                                                        n_mels=N_MELS, center=True, power=2.0)
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power")
        self.deltas = torchaudio.transforms.ComputeDeltas()

    def log_mel(self, wav):
        return self.to_db(self.mel(wav))                 # (batch, 64, frames), in dB

    def features(self, wav):
        db = self.log_mel(wav)
        d1 = self.deltas(db)
        d2 = self.deltas(d1)
        return torch.stack([db, d1, d2], dim=1)

    def forward(self, wav, lengths=None):
        x = self.features(wav)[..., :N_FRAMES]
        if lengths is not None:                          # fix the padded part of short clips
            for i in torch.nonzero(lengths < CROP_SAMPLES).flatten().tolist():
                f = self.features(wav[i:i + 1, :max(int(lengths[i]), N_FFT)])[0]
                t = min(f.shape[-1], N_FRAMES)
                x[i, 0] = f[0].min()
                x[i, 1:] = 0.0
                x[i, :, :, :t] = f[:, :, :t]
        return x


featurizer = LogMelDeltas().to(device).eval()
with torch.no_grad():
    INPUT_SHAPE = tuple(featurizer(torch.zeros(1, CROP_SAMPLES, device=device)).shape[1:])
print("Model input shape (channels, mel bands, frames):", INPUT_SHAPE)

In [ ]:
# Figure 3.2: Respiratory Distress compared with Non-distress (normal vocalization and environmental noise).
panels = [("Distress", "Respiratory Distress"),
          ("Normal", "Non-distress: normal vocalization"),
          ("Noise", "Non-distress: environmental noise")]
fig, axes = plt.subplots(len(panels), 2, figsize=(12, 7.5))
for r, (source, title) in enumerate(panels):
    idx = manifest.index[(manifest["role"] == "train/val/test") & (manifest["source"] == source)][0]
    wav, _ = crop(clips[idx])
    with torch.no_grad():
        mel = featurizer.log_mel(torch.tensor(wav, device=device).unsqueeze(0))[0].cpu().numpy()
    axes[r, 0].plot(np.arange(len(wav)) / SAMPLE_RATE, wav, linewidth=0.5)
    axes[r, 0].set_title(f"{title} (waveform)")
    axes[r, 0].set_ylabel("Amplitude")
    im = axes[r, 1].imshow(mel, origin="lower", aspect="auto", cmap="magma",
                           extent=[0, len(wav) / SAMPLE_RATE, 0, N_MELS])
    axes[r, 1].set_title(f"{title} (log-mel spectrogram)")
    axes[r, 1].set_ylabel("Mel band")
    fig.colorbar(im, ax=axes[r, 1], label="dB")
axes[-1, 0].set_xlabel("Time (s)")
axes[-1, 1].set_xlabel("Time (s)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figure_3_2_distress_vs_non_distress.png"), dpi=200)
plt.show()

## 5. Splitting and augmentation (Sections 3.1.4 and 3.1.2)

**Splitting.** Labels are encoded as 0 = Non-distress and 1 = Distress. The data is divided into 70% training, 15% validation and 15% test, stratified by source category. Clips cut from the **same source recording always stay in the same split**, so the test set never contains another slice of a recording the model was trained on. SmartEars file names carry a device ID and a start time (`rpi0001-20240103205100`), which identify the recording. Each seed produces a different split.

**Augmentation** is applied to the training split only, each time a clip is drawn:
- a random time shift of up to ±0.5 s (the features are rolled along the time axis, so no part of the clip is lost),
- masking of a random block of time frames, filled with each channel's own mean,
- masking of a random block of mel bands, filled the same way,
- a random loudness change of up to ±6 dB, added to the log-mel channel.

In [ ]:
def make_split(seed):
    """Returns {"train": indices, "val": indices, "test": indices} into `manifest`."""
    df = manifest.loc[main_idx]
    sgkf = StratifiedGroupKFold(n_splits=sum(SPLIT_FOLDS), shuffle=True, random_state=seed)
    fold_of = np.zeros(len(df), dtype=int)
    for fold, (_, idx) in enumerate(sgkf.split(df, df["source"], df[GROUP_BY])):
        fold_of[idx] = fold
    a, b = SPLIT_FOLDS[0], SPLIT_FOLDS[0] + SPLIT_FOLDS[1]
    split = {"train": main_idx[fold_of < a],
             "val": main_idx[(fold_of >= a) & (fold_of < b)],
             "test": main_idx[fold_of >= b]}
    groups = [set(manifest.loc[v, GROUP_BY]) for v in split.values()]
    assert not (groups[0] & groups[1]) and not (groups[0] & groups[2]) and not (groups[1] & groups[2]), "group leakage"
    return split


def split_table(split):
    rows = {}
    for name, idx in split.items():
        src = manifest.loc[idx, "source"].value_counts()
        rows[name] = {"Normal": src.get("Normal", 0), "Noise": src.get("Noise", 0),
                      "Non-distress": src.get("Normal", 0) + src.get("Noise", 0),
                      "Distress": src.get("Distress", 0), "total": len(idx)}
    table = pd.DataFrame(rows).T
    table["share"] = (table["total"] / table["total"].sum()).round(3)
    return table


print(f"Split for seed {PRIMARY_SEED} (grouped by {GROUP_BY}):")
print(split_table(make_split(PRIMARY_SEED)).to_string())

test_ids = {s: hashlib.md5(np.sort(make_split(s)["test"]).tobytes()).hexdigest()[:8] for s in SEEDS}
print("Test-set id per seed:", test_ids)
assert len(set(test_ids.values())) == len(SEEDS), "two seeds produced the same test set"

In [ ]:
class ClipDataset(Dataset):
    """Returns (5-s waveform, binary label, row index in `manifest`, real length in samples)."""

    def __init__(self, indices, train=False):
        self.indices = np.asarray(indices)
        self.labels = manifest["label"].to_numpy()
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        clip = clips[idx]
        start = None                                     # centre crop for validation and test
        if self.train and len(clip) > CROP_SAMPLES:      # random crop position for training
            start = int(torch.randint(0, len(clip) - CROP_SAMPLES + 1, (1,)))
        wav, n = crop(clip, start)
        return torch.from_numpy(wav), int(self.labels[idx]), idx, n


def augment_features(x):
    """Time shift, time masking, frequency masking and a loudness offset on a batch of feature tensors."""
    _, _, n_freq, n_time = x.shape
    max_shift = int(AUG_SHIFT_SEC * SAMPLE_RATE / HOP_LENGTH)
    for i in range(x.size(0)):
        if max_shift and random.random() < AUG_SHIFT_PROB:
            x[i] = torch.roll(x[i], random.randint(-max_shift, max_shift), dims=2)
        ch_mean = x[i].mean(dim=(1, 2), keepdim=True)    # each channel's own mean
        if random.random() < AUG_TIME_MASK_PROB:
            width = random.randint(5, n_time // 8)
            t0 = random.randint(0, n_time - width)
            x[i, :, :, t0:t0 + width] = ch_mean
        if random.random() < AUG_FREQ_MASK_PROB:
            height = random.randint(3, n_freq // 8)
            f0 = random.randint(0, n_freq - height)
            x[i, :, f0:f0 + height, :] = ch_mean
        if random.random() < AUG_GAIN_PROB:
            x[i, 0] += random.uniform(*AUG_GAIN_DB)      # a constant dB offset leaves the deltas unchanged
    return x


def make_loader(indices, train, seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(ClipDataset(indices, train), batch_size=BATCH_SIZE, shuffle=train,
                      num_workers=NUM_WORKERS, generator=g, pin_memory=(device.type == "cuda"))

## 6. Classification models (Section 3.2.1)

The model code below is the same as `test/sonara_models.py`, which the Raspberry Pi benchmark script imports, so weights saved by this notebook load there directly.

**CNN–BiLSTM (proposed).** Two 3 × 3 convolution layers with batch normalization, ReLU and pooling learn local time–frequency patterns, such as the broadband burst of a cough. Pooling turns the 5-second input into a sequence of 125 steps of about 40 ms each. The BiLSTM reads this sequence forward and backward and models how the local patterns are arranged across the 5 seconds, for example whether bursts repeat. The final forward state and the final backward state are joined, passed through dropout, and a fully connected layer outputs two logits.

**Comparison models. All are binary and use the same splits, seeds and training loop:**
- **CNN front end only**: the same convolution layers with the BiLSTM replaced by global average pooling. This is the ablation that shows what the BiLSTM adds.
- **DS-CNN**: a depthwise-separable CNN in the style used for TinyML keyword spotting (a reimplementation, not the team's earlier 3-class model).
- **MobileNetV3-Small**: a published architecture, trained from scratch on the same data.
- **Majority class**: always predicts Non-distress. Added to the results table as the floor.

In [ ]:
class CNN_BiLSTM(nn.Module):
    """CNN front end + bidirectional LSTM.

    pooling="final_states" (default): joins the final hidden state of the forward direction
        and of the backward direction, so both have read the whole sequence.
    pooling="last_step" (old V5 behaviour): takes the LSTM output at the last time step, where
        the backward direction has seen only one frame. Kept only to measure the effect of the fix.
    """

    def __init__(self, n_mels=64, num_classes=2, cnn_channels=64, lstm_hidden=128,
                 lstm_layers=1, dropout=DROPOUT, pooling="final_states"):
        super().__init__()
        self.pooling = pooling
        self.conv1 = nn.Conv2d(3, cnn_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(cnn_channels)
        self.conv2 = nn.Conv2d(cnn_channels, cnn_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(cnn_channels)
        self.pool = nn.MaxPool2d(kernel_size=(2, 2))
        self.lstm = nn.LSTM(input_size=cnn_channels * (n_mels // 4),    # 64 * 16 = 1024 inputs per time step
                            hidden_size=lstm_hidden, num_layers=lstm_layers,
                            batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        b, c, fq, t = x.shape                               # (B, 64, 16, T/4)
        x = x.permute(0, 3, 1, 2).reshape(b, t, c * fq)     # sequence of T/4 steps
        out, (h_n, _) = self.lstm(x)
        if self.pooling == "final_states":
            z = torch.cat([h_n[-2], h_n[-1]], dim=1)        # final forward + final backward state
        else:
            z = out[:, -1, :]
        return self.fc(self.dropout(z))


class CNNFrontEndOnly(nn.Module):
    """Ablation: the CNN–BiLSTM front end with global average pooling instead of the BiLSTM."""

    def __init__(self, n_mels=64, num_classes=2, cnn_channels=64, dropout=DROPOUT):
        super().__init__()
        self.conv1 = nn.Conv2d(3, cnn_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(cnn_channels)
        self.conv2 = nn.Conv2d(cnn_channels, cnn_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(cnn_channels)
        self.pool = nn.MaxPool2d(kernel_size=(2, 2))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(cnn_channels, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        return self.fc(self.dropout(self.gap(x).flatten(1)))


class _DSBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.dw = nn.Conv2d(ch, ch, kernel_size=3, padding=1, groups=ch, bias=False)
        self.bn_dw = nn.BatchNorm2d(ch)
        self.pw = nn.Conv2d(ch, ch, kernel_size=1, bias=False)
        self.bn_pw = nn.BatchNorm2d(ch)

    def forward(self, x):
        x = F.relu(self.bn_dw(self.dw(x)))
        return F.relu(self.bn_pw(self.pw(x)))


class DSCNN(nn.Module):
    """Depthwise-separable CNN in the style of Zhang et al. (2017), "Hello Edge"."""

    def __init__(self, num_classes=2, channels=172, n_blocks=4, dropout=DROPOUT):
        super().__init__()
        self.conv = nn.Conv2d(3, channels, kernel_size=(10, 4), stride=(2, 2), padding=(5, 1), bias=False)
        self.bn = nn.BatchNorm2d(channels)
        self.blocks = nn.Sequential(*[_DSBlock(channels) for _ in range(n_blocks)])
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(channels, num_classes)

    def forward(self, x):
        x = self.blocks(F.relu(self.bn(self.conv(x))))
        return self.fc(self.dropout(self.gap(x).flatten(1)))


def mobilenet_v3_small(num_classes=2):
    """MobileNetV3-Small from torchvision, trained from scratch (no ImageNet weights)."""
    from torchvision.models import mobilenet_v3_small as _mnv3
    return _mnv3(weights=None, num_classes=num_classes)


MODEL_BUILDERS = {
    "cnn_bilstm": lambda: CNN_BiLSTM(pooling="final_states"),
    "cnn_bilstm_laststep": lambda: CNN_BiLSTM(pooling="last_step"),
    "cnn_frontend_only": lambda: CNNFrontEndOnly(),
    "dscnn": lambda: DSCNN(),
    "mobilenetv3_small": lambda: mobilenet_v3_small(),
}
MODEL_LABELS = {
    "cnn_bilstm": "CNN-BiLSTM (proposed)",
    "cnn_bilstm_laststep": "CNN-BiLSTM (V5 last-step pooling)",
    "cnn_frontend_only": "CNN front end only (no LSTM)",
    "dscnn": "DS-CNN (binary)",
    "mobilenetv3_small": "MobileNetV3-Small",
}
PROPOSED = "cnn_bilstm"


def count_params(model):
    return sum(p.numel() for p in model.parameters())


for key in MODELS_TO_RUN:
    n = count_params(MODEL_BUILDERS[key]())
    print(f"{MODEL_LABELS[key]:34s} {n:>10,} parameters ({n * 4 / 1e6:.2f} MB as 32-bit floats)")

In [ ]:
# Where the CNN–BiLSTM parameters come from (panel row 15).
m = CNN_BiLSTM()
total = count_params(m)
print("CNN-BiLSTM parameter breakdown:")
for name, module in m.named_children():
    n = sum(p.numel() for p in module.parameters())
    if n:
        print(f"  {name:6s} {n:>10,}  {100.0 * n / total:5.1f}%")
print(f"  {'total':6s} {total:>10,}")
d, h = m.lstm.input_size, m.lstm.hidden_size
print(f"\nBiLSTM input size d = {d} (64 channels x {N_MELS // 4} mel bins), hidden size h = {h}")
print(f"2 directions x 4 gates x (h*d + h*h + 2h) = {2 * 4 * (h * d + h * h + 2 * h):,}")

## 7. Training objective and optimization (Sections 3.2.2 and 3.2.3)

- **Softmax:** $\hat{y}_c = e^{z_c} / \sum_{j=1}^{C} e^{z_j}$, where $z_c$ is the logit for class $c$ and $C = 2$. It runs inside the loss during training and is applied explicitly at inference to get the Distress probability.
- **Class weights (Equation 3-1):** $w_c = \bar{N} / N_c$, computed from the training clips of each split.
- **Label smoothing (Equation 3-3):** $\tilde{y}_c = y_c(1-\varepsilon) + \varepsilon / C$ with $\varepsilon = 0.02$, so the targets are 0.99 and 0.01 instead of 1 and 0. $\varepsilon$ is a training setting, not an error rate.
- **Weighted cross-entropy (Equation 3-2):** $\ell_i = -\sum_{c} w_c\, \tilde{y}_{i,c} \log \hat{y}_{i,c}$ per sample, and $L = \sum_i \ell_i / \sum_i w_{y_i}$ over a batch. This is PyTorch's `CrossEntropyLoss` with `weight` and `label_smoothing`.
- **Adam** with weight decay, a **scheduler** that halves the learning rate when the validation loss stops improving, and **early stopping** that keeps the weights from the epoch with the lowest validation loss.

The hyperparameters are the fixed values set in Part 1, the same for every model. They were not tuned by a search.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """One pass over a loader. Trains when an optimizer is given, otherwise evaluates."""
    training = optimizer is not None
    model.train(training)
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for wav, y, _, n in loader:
            wav, y = wav.to(device), y.to(device)
            with torch.no_grad():
                x = featurizer(wav, n)
                if training:
                    x = augment_features(x)
            logits = model(x)
            loss = criterion(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * y.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += y.size(0)
    return total_loss / total, correct / total


def make_criterion(train_idx, label_smoothing=LABEL_SMOOTHING):
    counts = np.bincount(manifest.loc[train_idx, "label"], minlength=2)
    weights = torch.tensor(counts.mean() / counts, dtype=torch.float32, device=device)
    return nn.CrossEntropyLoss(weight=weights, label_smoothing=label_smoothing), counts, weights


def train_model(key, seed, split, criterion):
    set_seed(seed)
    model = MODEL_BUILDERS[key]().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=1e-6)
    train_loader = make_loader(split["train"], True, seed)
    val_loader = make_loader(split["val"], False, seed)

    history, best_val_loss, best_state, stale = [], float("inf"), None, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion)
        scheduler.step(val_loss)
        improved = val_loss < best_val_loss - 1e-4
        if improved:
            best_val_loss, stale = val_loss, 0
            best_state = copy.deepcopy(model.state_dict())      # a real copy of the best weights
        else:
            stale += 1
        history.append({"epoch": epoch, "lr": optimizer.param_groups[0]["lr"],
                        "train_loss": train_loss, "train_acc": train_acc,
                        "val_loss": val_loss, "val_acc": val_acc, "best": improved})
        print(f"  [{key} seed {seed}] epoch {epoch:02d} train {train_loss:.4f} "
              f"val {val_loss:.4f} acc {val_acc:.4f}{' *' if improved else ''}")
        if stale >= EARLY_STOP_PATIENCE:
            break
    model.load_state_dict(best_state)
    return model.eval(), pd.DataFrame(history)


def predict(model, indices):
    """Distress probability (softmax) for every clip in `indices`."""
    model.eval()
    probs = []
    with torch.no_grad():
        for wav, _, _, n in make_loader(indices, False, 0):
            probs.append(F.softmax(model(featurizer(wav.to(device), n)), dim=1)[:, 1].cpu().numpy())
    return np.concatenate(probs)


def predict_file(model, clip):
    """Mean Distress probability over non-overlapping 5-s windows of one file (external test)."""
    starts = [0] if len(clip) <= CROP_SAMPLES else list(range(0, len(clip) - CROP_SAMPLES + 1, CROP_SAMPLES))
    if len(starts) > MAX_EXTERNAL_WINDOWS:
        keep = np.linspace(0, len(starts) - 1, MAX_EXTERNAL_WINDOWS).round().astype(int)
        starts = [starts[i] for i in keep]
    probs = []
    with torch.no_grad():
        for i in range(0, len(starts), 32):
            segs = [crop(clip, s) for s in starts[i:i + 32]]
            wav = torch.from_numpy(np.stack([s[0] for s in segs])).to(device)
            n = torch.tensor([s[1] for s in segs])
            probs.append(F.softmax(model(featurizer(wav, n)), dim=1)[:, 1].cpu())
    return torch.cat(probs).mean().item()


def choose_threshold(y_val, p_val, target=TARGET_RECALL):
    """Highest threshold whose validation distress recall still reaches the target."""
    ok = [t for t in np.round(np.arange(0.01, 0.995, 0.005), 3)
          if recall_score(y_val, (p_val >= t).astype(int), zero_division=0) >= target]
    return float(max(ok)) if ok else DEFAULT_THRESHOLD


def metrics_at(y_true, prob, threshold):
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    two = len(np.unique(y_true)) == 2
    return {"threshold": threshold, "n": int(len(y_true)), "accuracy": accuracy_score(y_true, pred),
            "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
            "distress_precision": precision_score(y_true, pred, zero_division=0),
            "distress_recall": recall_score(y_true, pred, zero_division=0),
            "distress_f1": f1_score(y_true, pred, zero_division=0),
            "fnr": fn / (fn + tp) if fn + tp else float("nan"),
            "fpr": fp / (fp + tn) if fp + tn else float("nan"),
            "roc_auc": roc_auc_score(y_true, prob) if two else float("nan"),
            "pr_auc": average_precision_score(y_true, prob) if two else float("nan"),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

## 8. Training over several seeds

For every seed the data is split again and every model is trained once on that split. Each finished run writes its metrics, predictions (with file paths), training history and best weights to `OUT_DIR/runs`. Finished runs are skipped when this cell is run again.

The external PVSD files are scored inside each run, with the threshold that was chosen on the SmartEars validation set.

In [ ]:
def run_paths(key, seed):
    base = os.path.join(RUN_DIR, f"{key}_seed{seed}")
    return {"metrics": base + ".json", "weights": base + ".pt", "preds": base + "_predictions.npz",
            "history": base + "_history.csv", "test_csv": base + "_test_predictions.csv",
            "ext_csv": base + "_pvsd_files.csv"}


y_all = manifest["label"].to_numpy()
for seed in SEEDS:
    split = make_split(seed)
    for key in MODELS_TO_RUN:
        paths = run_paths(key, seed)
        if os.path.exists(paths["metrics"]):
            print(f"skip (done): {key}_seed{seed}")
            continue
        t0 = time.time()
        criterion, train_counts, class_weights = make_criterion(split["train"])
        model, history = train_model(key, seed, split, criterion)
        p_val, p_test = predict(model, split["val"]), predict(model, split["test"])
        y_val, y_test = y_all[split["val"]], y_all[split["test"]]
        thr = choose_threshold(y_val, p_val)

        result = {"model": key, "label": MODEL_LABELS[key], "seed": seed, "params": count_params(model),
                  "test_set_id": test_ids[seed], "split_sizes": [len(v) for v in split.values()],
                  "best_epoch": int(history.loc[history["best"], "epoch"].max()), "epochs_run": len(history),
                  "train_minutes": (time.time() - t0) / 60, "train_counts": train_counts.tolist(),
                  "class_weights": class_weights.tolist(),
                  "val": metrics_at(y_val, p_val, thr), "test": metrics_at(y_test, p_test, thr),
                  "test_at_0_5": metrics_at(y_test, p_test, DEFAULT_THRESHOLD)}

        test_df = manifest.loc[split["test"], ["path", "recording", "original_label", "source", "label"]].copy()
        test_df["prob_distress"] = p_test
        test_df["pred"] = (p_test >= thr).astype(int)
        test_df["pred_at_0.5"] = (p_test >= DEFAULT_THRESHOLD).astype(int)
        test_df.to_csv(paths["test_csv"], index=False)
        result["test_by_source"] = test_df.groupby("source")["pred"].mean().round(4).to_dict()

        p_ext = np.array([predict_file(model, clips[i]) for i in external_idx])
        if len(external_idx):
            result["pvsd_file"] = metrics_at(y_all[external_idx], p_ext, thr)
            result["pvsd_file_at_0_5"] = metrics_at(y_all[external_idx], p_ext, DEFAULT_THRESHOLD)
            ext_df = manifest.loc[external_idx, ["path", "original_label", "source", "label", "seconds"]].copy()
            ext_df["prob_distress"] = p_ext
            ext_df["pred"] = (p_ext >= thr).astype(int)
            ext_df.to_csv(paths["ext_csv"], index=False)

        torch.save(model.state_dict(), paths["weights"])
        np.savez(paths["preds"], p_val=p_val, y_val=y_val, p_test=p_test, y_test=y_test,
                 rows_test=split["test"], p_ext=p_ext)
        history.to_csv(paths["history"], index=False)
        with open(paths["metrics"], "w") as f:      # written last: marks the run as complete
            json.dump(result, f, indent=1, default=float)
        t = result["test"]
        print(f"DONE {key}_seed{seed}: thr {thr:.3f} | test acc {t['accuracy']:.4f} recall {t['distress_recall']:.4f} "
              f"F1 {t['distress_f1']:.4f} AUC {t['roc_auc']:.4f} | {result['train_minutes']:.1f} min\n")

In [ ]:
# Training and validation curves for the primary seed.
fig, axes = plt.subplots(2, len(MODELS_TO_RUN), figsize=(4.6 * len(MODELS_TO_RUN), 6.5), squeeze=False)
for col, key in enumerate(MODELS_TO_RUN):
    hist = pd.read_csv(run_paths(key, PRIMARY_SEED)["history"])
    best_epoch = int(hist.loc[hist["best"], "epoch"].max())
    for row, metric in enumerate(["loss", "acc"]):
        ax = axes[row, col]
        ax.plot(hist["epoch"], hist[f"train_{metric}"], label="train")
        ax.plot(hist["epoch"], hist[f"val_{metric}"], label="validation")
        ax.axvline(best_epoch, color="gray", linestyle="--", linewidth=1, label="best checkpoint")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss" if metric == "loss" else "Accuracy")
        ax.set_title(MODEL_LABELS[key] if row == 0 else "")
        ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=200)
plt.show()

## 9. Evaluation on the held-out test set (Sections 3.2.4 and 3.3.5)

Every model is evaluated on the test set of its split, which was not used for training, model selection or threshold selection. Metrics are computed per clip with Distress as the positive class, and reported as **mean ± standard deviation over the seeds**.

- **Main table (`summary_table.csv`)**: at the recall-target threshold, the highest threshold that still reaches the target distress recall on the *validation* set. Missing distress is the costly error, so the threshold is set from recall, not from accuracy.
- **Second table**: the same models at threshold 0.5.
- **Reference points**: the majority-class row, the CNN front end without the LSTM, and MobileNetV3-Small. The Proportional Chance Criterion is printed only as a minimum check.

In [ ]:
def load_run(key, seed):
    paths = run_paths(key, seed)
    with open(paths["metrics"]) as f:
        return json.load(f), np.load(paths["preds"])


def load_model(key, seed):
    model = MODEL_BUILDERS[key]()
    model.load_state_dict(torch.load(run_paths(key, seed)["weights"], map_location="cpu"))
    return model.to(device).eval()


def pm(series, digits=3):
    if len(series) > 1:
        return f"{series.mean():.{digits}f} ± {series.std(ddof=1):.{digits}f}"
    return f"{series.mean():.{digits}f}"


flat = pd.json_normalize([load_run(k, s)[0] for k in MODELS_TO_RUN for s in SEEDS], sep=".")
flat.to_csv(os.path.join(OUT_DIR, "all_runs.csv"), index=False)
HAS_EXTERNAL = "pvsd_file.roc_auc" in flat.columns

cols = {"test.accuracy": "Accuracy", "test.macro_f1": "Macro-F1", "test.distress_recall": "Distress recall",
        "test.distress_precision": "Distress precision", "test.distress_f1": "Distress F1",
        "test.fnr": "FNR", "test.fpr": "FPR", "test.roc_auc": "ROC-AUC"}
if HAS_EXTERNAL:
    cols.update({"pvsd_file.roc_auc": "PVSD ROC-AUC (file)", "pvsd_file.distress_recall": "PVSD recall (file)",
                 "pvsd_file.fpr": "PVSD FPR (file)"})

# Majority class (always Non-distress), on the same test sets.
maj = pd.DataFrame([metrics_at(y_all[make_split(s)["test"]], np.zeros(len(make_split(s)["test"])), 0.5) for s in SEEDS])
summary = [{"Model": "Majority class (always Non-distress)", "Parameters": "0", "Seeds": len(SEEDS),
            "Accuracy": pm(maj["accuracy"]), "Macro-F1": pm(maj["macro_f1"]), "Distress recall": "0.000",
            "Distress F1": "0.000", "FNR": "1.000", "FPR": "0.000", "ROC-AUC": "0.500"}]
for key in MODELS_TO_RUN:
    g = flat[flat["model"] == key]
    row = {"Model": MODEL_LABELS[key], "Parameters": f"{int(g['params'].iloc[0]):,}", "Seeds": len(g),
           "Epochs (best)": pm(g["best_epoch"], 1), "Threshold": pm(g["test.threshold"])}
    row.update({name: pm(g[c]) for c, name in cols.items()})
    summary.append(row)
summary = pd.DataFrame(summary)[["Model", "Parameters", "Seeds", "Epochs (best)", "Threshold"] + list(cols.values())]
summary.to_csv(os.path.join(OUT_DIR, "summary_table.csv"), index=False)
print(f"New Table 3.1: test results at the recall-target threshold (validation distress recall >= {TARGET_RECALL})")
print(summary.fillna("").to_string(index=False))

cols05 = {"test_at_0_5.accuracy": "Accuracy", "test_at_0_5.macro_f1": "Macro-F1",
          "test_at_0_5.distress_recall": "Distress recall", "test_at_0_5.distress_precision": "Distress precision",
          "test_at_0_5.distress_f1": "Distress F1"}
table05 = pd.DataFrame([{"Model": MODEL_LABELS[k], **{n: pm(flat.loc[flat["model"] == k, c]) for c, n in cols05.items()}}
                        for k in MODELS_TO_RUN])
table05.to_csv(os.path.join(OUT_DIR, "summary_table_threshold_0_5.csv"), index=False)
print("\nSame models at threshold 0.5:")
print(table05.to_string(index=False))

In [ ]:
# Paired comparison: same seed = same split, so compare the CNN–BiLSTM with each other model seed by seed.
def paired(a, b, metric):
    A = flat[flat["model"] == a].set_index("seed")[metric]
    B = flat[flat["model"] == b].set_index("seed")[metric]
    d = A - B
    return {"pair": f"{a} - {b}", "metric": metric, "seeds": len(d), "mean_diff": round(d.mean(), 4),
            "sd_diff": round(d.std(ddof=1), 4) if len(d) > 1 else None, "wins": int((d > 0).sum())}


pair_metrics = ["test.distress_recall", "test.distress_f1", "test.roc_auc"] + (["pvsd_file.roc_auc"] if HAS_EXTERNAL else [])
pairs = pd.DataFrame([paired(PROPOSED, b, m) for b in MODELS_TO_RUN if b != PROPOSED for m in pair_metrics])
pairs.to_csv(os.path.join(OUT_DIR, "paired_comparison.csv"), index=False)
print(pairs.to_string(index=False))
print("\nRead 'wins' as: on how many seeds the CNN-BiLSTM scored higher. With 5 seeds, a 3-2 split is "
      "not evidence of a difference; report it as no clear difference.")

# Which Non-distress clips cause false alarms? Share of test clips flagged as Distress, by source category.
by_source = pd.DataFrame([{"model": MODEL_LABELS[r["model"]], "seed": r["seed"], "source": s, "flagged": v}
                          for r in (load_run(k, s)[0] for k in MODELS_TO_RUN for s in SEEDS)
                          for s, v in r["test_by_source"].items()])
print("\nShare of test clips flagged as Distress (for Distress this is recall; for Normal and Noise it is the false-alarm rate):")
print(by_source.groupby(["model", "source"])["flagged"].agg(["mean", "std"]).round(3).to_string())

In [ ]:
# Model size and CPU latency. This is the Colab CPU, NOT the Raspberry Pi 4: for the manuscript,
# copy test/sonara_models.py and test/pi_latency_benchmark.py to the Pi and run the script there.
def time_ms(fn, runs=30):
    for _ in range(5):
        fn()
    ts = []
    for _ in range(runs):
        t = time.perf_counter()
        fn()
        ts.append((time.perf_counter() - t) * 1000)
    return round(float(np.median(ts)), 1), round(float(np.percentile(ts, 95)), 1)


cpu_feat = LogMelDeltas().eval()
wav_cpu, x_cpu = torch.randn(1, CROP_SAMPLES) * 0.1, torch.randn(1, 3, N_MELS, N_FRAMES)
with torch.no_grad():
    lat = [{"step": "feature extraction (5-s clip)", "params": "", "size_mb": "",
            **dict(zip(["cpu_median_ms", "cpu_p95_ms"], time_ms(lambda: cpu_feat(wav_cpu))))}]
    for key in MODELS_TO_RUN:
        mdl = MODEL_BUILDERS[key]().eval()
        n = count_params(mdl)
        lat.append({"step": MODEL_LABELS[key], "params": n, "size_mb": round(n * 4 / 1e6, 2),
                    **dict(zip(["cpu_median_ms", "cpu_p95_ms"], time_ms(lambda: mdl(x_cpu))))})
latency = pd.DataFrame(lat)
latency.to_csv(os.path.join(OUT_DIR, "latency_proxy_colab_cpu.csv"), index=False)
print(f"CPU threads: {torch.get_num_threads()} (Colab proxy, NOT the Pi)")
print(latency.to_string(index=False))

In [ ]:
# Detailed results for the proposed model on the primary seed.
result, preds = load_run(PROPOSED, PRIMARY_SEED)
y_test, p_test, rows_test = preds["y_test"], preds["p_test"], preds["rows_test"]
TUNED_THRESHOLD = result["test"]["threshold"]
pred_tuned = (p_test >= TUNED_THRESHOLD).astype(int)
print(f"{MODEL_LABELS[PROPOSED]}, seed {PRIMARY_SEED}, threshold {TUNED_THRESHOLD:.3f}")
print(classification_report(y_test, pred_tuned, target_names=CLASS_NAMES, digits=3))

# Proportional Chance Criterion (Equations 3-12 and 3-13): a minimum check only.
q = y_test.mean()
p = 1.0 - q
pcc = p ** 2 + q ** 2
test_acc = accuracy_score(y_test, pred_tuned)
print(f"Test proportions: Non-distress p = {p:.3f}, Distress q = {q:.3f}")
print(f"PCC = p² + q² = {pcc:.3f} | 1.25 × PCC = {1.25 * pcc:.3f} | majority-class accuracy = {max(p, q):.3f}")
print(f"Test accuracy {test_acc:.3f} -> {'exceeds' if test_acc > 1.25 * pcc else 'does NOT exceed'} 1.25 × PCC")

In [ ]:
# Confusion matrices (recall-target threshold) and ROC curves, primary seed.
fig, axes = plt.subplots(1, len(MODELS_TO_RUN) + 1, figsize=(4.4 * (len(MODELS_TO_RUN) + 1), 4.2))
for ax, key in zip(axes, MODELS_TO_RUN):
    r, pr = load_run(key, PRIMARY_SEED)
    cm = confusion_matrix(pr["y_test"], (pr["p_test"] >= r["test"]["threshold"]).astype(int), labels=[0, 1])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=13)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(CLASS_NAMES, rotation=90, va="center")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{MODEL_LABELS[key]}\n(threshold {r['test']['threshold']:.3f})", fontsize=9)
    fpr, tpr, _ = roc_curve(pr["y_test"], pr["p_test"])
    axes[-1].plot(fpr, tpr, label=f"{MODEL_LABELS[key]} (AUC {r['test']['roc_auc']:.3f})")
axes[-1].plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1)
axes[-1].set_xlabel("False positive rate")
axes[-1].set_ylabel("True positive rate")
axes[-1].set_title("ROC curves")
axes[-1].legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrices_and_roc.png"), dpi=200)
plt.show()

# Model file for the gateway, with everything needed to repeat the preprocessing.
proposed_model = load_model(PROPOSED, PRIMARY_SEED)
torch.save({"state_dict": proposed_model.state_dict(), "class_names": CLASS_NAMES,
            "threshold_recall_target": TUNED_THRESHOLD, "threshold_default": DEFAULT_THRESHOLD,
            "preprocessing": {"sample_rate": SAMPLE_RATE, "crop_sec": CROP_SEC, "n_fft": N_FFT,
                              "hop_length": HOP_LENGTH, "n_mels": N_MELS, "scale": "power in dB + delta + delta-delta"}},
           os.path.join(OUT_DIR, "sonara_cnn_bilstm.pt"))
print("Saved sonara_cnn_bilstm.pt. Per-clip test predictions with file paths are in", RUN_DIR)

## 10. External test on PVSD

PVSD is never used for training or for choosing the threshold. Each PVSD file is cut into non-overlapping 5-second windows (at most 50, evenly spaced), the Distress probabilities of the windows are averaged, and the file gets one prediction. This shows how a model trained on SmartEars behaves on recordings from a different source.

Read these numbers with care: PVSD labels come from a treated versus untreated group of birds, not from a per-clip review, so a "Healthy" file can still contain respiratory sounds and the reverse.

In [ ]:
if not HAS_EXTERNAL:
    print("No external dataset was found when the models were run. Add a PVSD folder "
          "(Healthy / Unhealthy / Noise) to the repo, delete OUT_DIR/runs and run again.")
else:
    y_ext = y_all[external_idx]
    ext_cols = {"pvsd_file.accuracy": "Accuracy", "pvsd_file.macro_f1": "Macro-F1",
                "pvsd_file.distress_recall": "Distress recall", "pvsd_file.distress_precision": "Distress precision",
                "pvsd_file.fpr": "FPR", "pvsd_file.roc_auc": "ROC-AUC", "pvsd_file.pr_auc": "PR-AUC"}
    external_table = pd.DataFrame([{"Model": MODEL_LABELS[k],
                                    **{n: pm(flat.loc[flat["model"] == k, c]) for c, n in ext_cols.items()}}
                                   for k in MODELS_TO_RUN])
    external_table.to_csv(os.path.join(OUT_DIR, "external_pvsd_table.csv"), index=False)
    print(f"External test: {len(external_idx)} PVSD files, Non-distress {int((y_ext == 0).sum())} / "
          f"Distress {int((y_ext == 1).sum())}; per file, at each run's recall-target threshold")
    print(external_table.to_string(index=False))

    ext_df = pd.read_csv(run_paths(PROPOSED, PRIMARY_SEED)["ext_csv"])
    print(f"\n{MODEL_LABELS[PROPOSED]}, seed {PRIMARY_SEED}: share of files classified as Distress, by original label")
    print(ext_df.groupby("original_label")["pred"].agg(files="size", flagged="mean").round(3).to_string())
    short = ext_df["seconds"] < CROP_SEC
    print(f"\nFiles shorter than {CROP_SEC:.0f} s: {int(short.sum())} | flagged {ext_df.loc[short, 'pred'].mean():.3f} "
          f"vs {ext_df.loc[~short, 'pred'].mean():.3f} for longer files")

## 11. Gateway inference and zone-level attribution (Sections 3.3.2 to 3.3.4)

This part implements the gateway logic: parse a packet, preprocess the audio the same way as in training, classify it, apply the decision threshold, look up the zone of the sending node, log the event, and raise a zone alert when the share of Distress detections in a moving window passes a threshold.

**This is a software simulation.** Held-out test clips stand in for audio packets, and the node assignment is made up. It checks that the gateway logic works and shows how clip-level errors carry over to zone-level alerts. It is **not** the controlled playback test or the field comparison with veterinary observation, which need the physical nodes. It also says nothing about whether 5 seconds of audio can be sent over the LoRa link in time; that depends on the link decision.

In [ ]:
NODE_TO_ZONE = {"node-01": "Zone 1", "node-02": "Zone 2", "node-03": "Zone 3"}
ALERT_WINDOW = 20         # number of most recent clips considered per zone
ALERT_RATE = 0.40         # alert when this share of them is classified as Distress
PACKET_INTERVAL_MIN = 0.5
GATEWAY_THRESHOLD = TUNED_THRESHOLD


class Gateway:
    def __init__(self, model, node_to_zone, threshold):
        self.model = model.eval()
        self.node_to_zone = node_to_zone
        self.threshold = threshold
        self.recent = {zone: deque(maxlen=ALERT_WINDOW) for zone in node_to_zone.values()}
        self.alert_active = {zone: False for zone in node_to_zone.values()}
        self.event_log, self.alerts = [], []

    def process_packet(self, packet):
        # 1. Parse the packet.
        node_id, audio, sr, t_min = packet["node_id"], packet["audio"], packet["sample_rate"], packet["t_min"]
        # 2. Same preprocessing as training: mono, 16 kHz, 5-s input.
        audio = np.asarray(audio, dtype=np.float32)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != SAMPLE_RATE:
            g = gcd(int(sr), SAMPLE_RATE)
            audio = resample_poly(audio, SAMPLE_RATE // g, sr // g).astype(np.float32)
        wav, n = crop(np.clip(audio * 32767.0, -32768, 32767).astype(np.int16))
        # 3-4. Three-channel tensor -> CNN–BiLSTM -> class probabilities.
        with torch.no_grad():
            x = featurizer(torch.from_numpy(wav).unsqueeze(0).to(device), torch.tensor([n]))
            prob = F.softmax(self.model(x), dim=1)[0, 1].item()
        # 5. Decision threshold on the Distress probability.
        pred = int(prob >= self.threshold)
        # Zone attribution: fixed node-to-zone lookup (Section 3.3.3).
        zone = self.node_to_zone[node_id]
        # Zone-level aggregation and alert.
        self.recent[zone].append(pred)
        rate = sum(self.recent[zone]) / len(self.recent[zone])
        full = len(self.recent[zone]) == ALERT_WINDOW
        if full and rate >= ALERT_RATE and not self.alert_active[zone]:
            self.alert_active[zone] = True
            self.alerts.append({"t_min": t_min, "zone": zone, "distress_rate": rate})
        elif full and rate < ALERT_RATE:
            self.alert_active[zone] = False
        event = {"t_min": t_min, "node_id": node_id, "zone": zone, "predicted_class": CLASS_NAMES[pred],
                 "prob_distress": round(prob, 4), "zone_distress_rate": round(rate, 3)}
        self.event_log.append(event)
        return event

In [ ]:
# Simulated scenario: three zones. Zones 1 and 3 stay healthy. Zone 2 is healthy for the
# first half, then 60% of its clips contain respiratory distress.
rng = np.random.default_rng(PRIMARY_SEED)
pool = {0: rows_test[y_test == 0], 1: rows_test[y_test == 1]}
PACKETS_PER_ZONE = 120
OUTBREAK_ZONE, OUTBREAK_START, OUTBREAK_SHARE = "Zone 2", 60, 0.60

gateway = Gateway(proposed_model, NODE_TO_ZONE, GATEWAY_THRESHOLD)
truth = []
for k in range(PACKETS_PER_ZONE):
    for node_id, zone in NODE_TO_ZONE.items():
        distress = int(zone == OUTBREAK_ZONE and k >= OUTBREAK_START and rng.random() < OUTBREAK_SHARE)
        clip = clips[int(rng.choice(pool[distress]))]
        gateway.process_packet({"node_id": node_id, "audio": clip.astype(np.float32) / 32768.0,
                                "sample_rate": SAMPLE_RATE, "t_min": k * PACKET_INTERVAL_MIN})
        truth.append(distress)

event_log = pd.DataFrame(gateway.event_log)
event_log["true_class"] = [CLASS_NAMES[t] for t in truth]
event_log.to_csv(os.path.join(OUT_DIR, "simulated_gateway_event_log.csv"), index=False)
print(f"Gateway threshold: {GATEWAY_THRESHOLD:.3f}. Event log (first rows):")
print(event_log.head(6).to_string(index=False))

In [ ]:
is_true = event_log["true_class"] == "Distress"
is_pred = event_log["predicted_class"] == "Distress"
zone_summary = pd.DataFrame({
    "clips": event_log.groupby("zone").size(),
    "true Distress": is_true.groupby(event_log["zone"]).sum(),
    "detected Distress": is_pred.groupby(event_log["zone"]).sum(),
    "false alarms": (is_pred & ~is_true).groupby(event_log["zone"]).sum(),
    "missed": (~is_pred & is_true).groupby(event_log["zone"]).sum(),
})
alerts = pd.DataFrame(gateway.alerts, columns=["t_min", "zone", "distress_rate"])
zone_summary["alerts raised"] = alerts.groupby("zone").size().reindex(zone_summary.index).fillna(0).astype(int)
zone_summary["first alert (min)"] = alerts.groupby("zone")["t_min"].min().reindex(zone_summary.index)
print(f"Simulated outbreak: {OUTBREAK_ZONE} from minute {OUTBREAK_START * PACKET_INTERVAL_MIN:.0f}")
print(zone_summary.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
for zone, g in event_log.groupby("zone"):
    ax.plot(g["t_min"], g["zone_distress_rate"], label=zone)
ax.axhline(ALERT_RATE, color="red", linestyle="--", linewidth=1, label="alert threshold")
ax.axvline(OUTBREAK_START * PACKET_INTERVAL_MIN, color="gray", linestyle=":", label=f"outbreak starts ({OUTBREAK_ZONE})")
ax.set_xlabel("Simulated time (minutes)")
ax.set_ylabel(f"Share of last {ALERT_WINDOW} clips classified as Distress")
ax.set_title("Zone-level distress rate (simulation)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "simulated_zone_distress_rate.png"), dpi=200)
plt.show()

## 12. Optional label-smoothing check, and values for the manuscript

**Label smoothing (panel rows 7 and 8).** The first cell compares ε = 0, 0.02, 0.05 and 0.1 for the CNN–BiLSTM on **validation** data only; the test set is not used to choose settings. It needs 12 extra training runs. Set `RUN_LS_ABLATION = True` to run it.

**Manuscript values.** The second cell prints the facts and numbers that Sections 3.1 and 3.2 need. All figures, tables and CSV files are in `OUT_DIR`.

In [ ]:
RUN_LS_ABLATION = False
if RUN_LS_ABLATION:
    ls_rows = []
    for eps in [0.0, 0.02, 0.05, 0.1]:
        for seed in SEEDS[:3]:
            split = make_split(seed)
            criterion, _, _ = make_criterion(split["train"], label_smoothing=eps)
            model, _ = train_model(PROPOSED, seed, split, criterion)
            p_val, y_val = predict(model, split["val"]), y_all[split["val"]]
            mv = metrics_at(y_val, p_val, choose_threshold(y_val, p_val))
            ls_rows.append({"epsilon": eps, "seed": seed, "val_auc": mv["roc_auc"],
                            "val_distress_f1": mv["distress_f1"], "val_recall": mv["distress_recall"]})
    ls = pd.DataFrame(ls_rows)
    ls.to_csv(os.path.join(OUT_DIR, "label_smoothing_ablation.csv"), index=False)
    print(ls.groupby("epsilon")[["val_auc", "val_distress_f1", "val_recall"]].agg(["mean", "std"]).round(4).to_string())
    print("Validation loss is not compared across epsilon values, because smoothing changes the loss itself.")

In [ ]:
split = make_split(PRIMARY_SEED)
print("Section 3.1")
print(f"  Training datasets: {', '.join(TRAIN_DATASETS)} ({len(main_idx):,} clips: "
      f"{manifest.loc[main_idx, 'source'].value_counts().to_dict()}). External test: {len(external_idx):,} PVSD files.")
print(f"  Input: {SAMPLE_RATE} Hz mono, one {CROP_SEC:.0f}-s clip per file, {N_MELS} mel bands "
      f"({N_FFT / SAMPLE_RATE * 1000:.0f} ms window, {HOP_LENGTH / SAMPLE_RATE * 1000:.0f} ms hop), power in dB, "
      f"plus delta and delta-delta -> {INPUT_SHAPE[0]} x {INPUT_SHAPE[1]} x {INPUT_SHAPE[2]}.")
print(f"  Split: 70/15/15 by {GROUP_BY}, stratified by source category; clips per split (seed {PRIMARY_SEED}): "
      f"{ {k: len(v) for k, v in split.items()} }")
print(f"  Augmentation (training only): time shift up to ±{AUG_SHIFT_SEC} s, p={AUG_SHIFT_PROB}; time mask p={AUG_TIME_MASK_PROB}; "
      f"frequency mask p={AUG_FREQ_MASK_PROB}; loudness offset of {AUG_GAIN_DB[0]:+.0f} to {AUG_GAIN_DB[1]:+.0f} dB, p={AUG_GAIN_PROB}.")
print("Section 3.2.1")
print(f"  Hyperparameters were set by hand and not tuned: Adam, learning rate {LEARNING_RATE}, "
      f"weight decay {WEIGHT_DECAY}, batch size {BATCH_SIZE}, dropout {DROPOUT}.")
print(f"  Maximum {MAX_EPOCHS} epochs; early stopping patience {EARLY_STOP_PATIENCE}; {len(SEEDS)} seeds: {SEEDS}.")
for key in MODELS_TO_RUN:
    g = flat[flat["model"] == key]
    print(f"  {MODEL_LABELS[key]}: {int(g['params'].iloc[0]):,} parameters; stopped after "
          f"{int(g['epochs_run'].min())} to {int(g['epochs_run'].max())} epochs.")
print("  Inference time: see latency_proxy_colab_cpu.csv (Colab CPU). Measure on the Raspberry Pi 4 with test/pi_latency_benchmark.py.")
print("Section 3.2.2")
print(f"  Label smoothing factor {LABEL_SMOOTHING}, set by hand (not selected on the validation set).")
print("Section 3.3.2")
print(f"  Decision threshold: highest value with validation distress recall >= {TARGET_RECALL}; "
      f"mean over seeds for the CNN-BiLSTM = {flat.loc[flat['model'] == PROPOSED, 'test.threshold'].mean():.3f}. "
      f"Results at {DEFAULT_THRESHOLD} are in summary_table_threshold_0_5.csv.")